# 03.2 Advanced Chromatography: Exercises

## Contents

**Tools:** CADET-Process (Python); CADET-GUI flow sheet diagram.

**Python used:** Unit creation, flow-sheet connections, model builders, simulation and plotting.

**Practises:** instrument dispersion, `CarouselBuilder` zones, salt-dependent capture/elution and collected protein composition.

1. Valves and tubing
2. Bind–elute carousel

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

```{tip}
Run the cell below once with **Shift+Enter** to load the shared model setup, plotting and collection functions for this exercise.
```

In [ ]:
# Show notebook plots and load the shared column setup and collection plots.
%matplotlib inline
%run resources/advanced_chromatography.py

## Exercise 1: modeling dispersion of valves and tubing

The supplied `valve_process()` returns Example 1 from the lesson: feed and eluent → valve (`Cstr`) → Langmuir column → outlet. Add tubing between the valve and the column, using a `TubularReactor` with $L = 0.5~\text{m}$, cross-section $A = 1 \times 10^{-5}~\text{m}^2$ and $D_{\text{ax}} = 1 \times 10^{-5}~\text{m}^2~\text{s}^{-1}$.

```{tip}
Run the cell below once with **Shift+Enter**. It creates two independent copies of the lesson model: `valve_only` stays unchanged for comparison; you add the tubing to `tubing_process`.
```

In [ ]:
# Two independent copies of the lesson's valve model.
valve_only = valve_process()
tubing_process = valve_process()
flow_sheet = tubing_process.flow_sheet
component_system = tubing_process.component_system

**Task:** in `flow_sheet`, create the unit `tubing`, add it, remove the direct connection valve → column and connect valve → tubing → column. The existing units are available as `flow_sheet.valve` and `flow_sheet.column`.

```{tip}
**Methods for this exercise**

- `TubularReactor(component_system, "tubing")` with `length`, `cross_section_area` and `axial_dispersion`. See [`TubularReactor`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.TubularReactor.html).
- `flow_sheet.add_unit(unit)`, `flow_sheet.remove_connection(origin, destination)` and `flow_sheet.add_connection(origin, destination)`. See the [flow sheet guide](https://cadet-process.readthedocs.io/en/latest/user_guide/process_model/flow_sheet.html).
```

In [ ]:
# Add the tubing and rewire the connections


The supplied cell simulates both models and compares the column inlet and outlet signals. Which changes come from the added tubing?

In [ ]:
# Simulate both models and overlay their column inlet and outlet signals.
import matplotlib.pyplot as plt
from CADETProcess.simulator import Cadet

simulator = Cadet()
valve_results = simulator.simulate(valve_only)
tubing_results = simulator.simulate(tubing_process)

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, position in zip(axes, ["inlet", "outlet"]):
    for results, label, style in [(valve_results, "valve only", "--"), (tubing_results, "valve + tubing", "-")]:
        signal = getattr(results.solution.column, position)
        # Solution columns follow the component system: 0 = A, 1 = B
        for index, name in enumerate(["A", "B"]):
            ax.plot(signal.time / 60, signal.solution[:, index], linestyle=style,
                    color=f"C{index}", label=f"{name}, {label}")
    ax.set(title=f"Column {position}", xlabel="Time / min")
axes[0].set_ylabel("Concentration / mM")
axes[1].legend()
fig.tight_layout()
plt.show()

```{dropdown} Interpretation
The tubing holds $0.5 \times 10^{-5}~\text{m}^3 = 5~\text{mL}$, so at 1 mL/min it delays the pulse by about 5 min before it reaches the column. Its dispersion also broadens the pulse, so the column receives a lower, wider feed. Both effects carry through to the outlet: assigning them to the column would distort its fitted transport or binding parameters.
```

## Exercise 2: continuous capture and elution

Purify **Target** from a weakly binding **Impurity**. At **50 mM salt**, Target binds to the resin while Impurity stays mainly in the liquid. **500 mM salt** releases Target. This applies the SMA binding principle from Simple Chromatography to a carousel.

```{figure} ./resources/bind_elute_carousel.svg
:width: 85%
```

Columns move through four zones in the model; the connections switch every **5 min**:

| Zone | Columns | Purpose |
|---|---|---|
| Wash | 3 in series | Restore low-salt conditions before loading |
| Feed | 3 in parallel | Capture Target at low salt |
| Dilute | 2 in series, reverse flow | Flush out unbound Impurity before elution |
| Elute | 2 in series | Release Target with high salt into Product |

All three inlets supply **1.5 mL/min**. Feed contains **0.02 mM Impurity and 0.02 mM Target** in **50 mM salt**; Wash is **50 mM salt** without protein; Elute is **500 mM salt** without protein. Washing and feed liquid mix before Dilute, reducing the unbound protein concentration.

```{tip}
Run the cell below once with **Shift+Enter** to load an independent **10 × 1 cm** column with porosity **0.4** and salt-dependent binding. The [setup script](resources/advanced_chromatography.py) contains its parameters. This template does not reuse the Langmuir column from Exercise 1.
```

In [ ]:
# Load the prebuilt column template for this bind–elute carousel.
carousel_component_system, carousel_column = carousel_column_template()

In [ ]:
# Inlets and outlets


Create the four zones with the column counts in the table. Set `flow_direction=-1` in Dilute for reverse flow.

In [ ]:
# Four zones


Connect the zones in the builder. Send **99%** of the Dilute outlet to Waste and **1%** onwards to Elute. The retained Target travels into Elute with the column; most dissolved Impurity leaves through Waste.

In [ ]:
# Builder with connections


The zone diagram shows the configured connections before the builder expands the zones into individual columns.

In [ ]:
# Show the configured carousel zones and their inlet/outlet connections.
from IPython.display import SVG, display
from cadetgui.widgets.diagrams.flow_sheet import render_flow_sheet_svg

diagram = render_flow_sheet_svg(builder.flow_sheet)
display(SVG(diagram[diagram.index("<svg"):diagram.rindex("</svg>") + len("</svg>")]))

Build the process with a **5 min** switch time. With ten columns, one complete cycle lasts **50 min**. Run **two cycles**, then collect the last full cycle; the first cycle starts with unloaded columns.

In [ ]:
# Build the connected carousel and simulate two complete cycles.
from CADETProcess.simulator import Cadet
builder.switch_time = 5 * 60
carousel_process = builder.build_process()
carousel_simulator = Cadet()
carousel_simulator.n_cycles = 2
carousel_results = carousel_simulator.simulate(carousel_process)

In [ ]:
# Compare the outlet proteins; buffer salt is omitted so the protein signals remain visible.
import matplotlib.pyplot as plt

fig, ax = carousel_results.solution.o_product.inlet.plot(components=["Impurity", "Target"])
ax.set_title("Product: eluted Target")
fig, ax = carousel_results.solution.o_waste.inlet.plot(components=["Impurity", "Target"])
ax.set_title("Waste: unbound Impurity")
plt.show()

### What is collected?

Collect Product and Waste over the **last complete cycle**. `create_fraction()` integrates concentration times flow rate and the collected liquid volume. Report amounts in **µmol** and protein composition as **Target / (Impurity + Target)**; salt is excluded from this composition calculation.

In [ ]:
# Pool both outlet streams and the feed over the same complete cycle.
product_signal = carousel_results.solution_cycles.o_product.inlet[-1]
waste_signal = carousel_results.solution_cycles.o_waste.inlet[-1]
carousel_feed_signal = carousel_results.solution_cycles.i_feed.outlet[-1]

product_pool = product_signal.create_fraction()
waste_pool = waste_signal.create_fraction()
carousel_feed_pool = carousel_feed_signal.create_fraction()
_ = show_carousel_collection(product_pool, waste_pool, carousel_feed_pool)

Compare the pools with the **50:50 protein feed**. Does Product contain mostly Target, and Waste mostly Impurity? The amount plot also checks whether a high target share is accompanied by a useful collected amount.

```{dropdown} Interpretation
Low salt retains Target during loading and rinsing. Switching the column into the high-salt Elute zone releases it into Product; most unbound Impurity leaves through Waste. The supplied settings give a clear collected separation. These illustrative parameters are a demonstration, not an optimized method; two cycles alone do not prove cyclic stationarity.
```

**Next lesson:** [**04.1 Optimization Essentials**](../../04%20Characterization%20and%20Optimization/04.1%20Optimization%20Essentials/04.1a_optimization_essentials_lesson.ipynb)